In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("Libraries loaded!")

## 14.1 GridWorld Environment

In [ ]:
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.goal = (size-1, size-1)
        self.obstacles = [(1, 1), (2, 2), (3, 1)]
        self.reset()
        
        # Actions: up, down, left, right
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.action_names = ['Up', 'Down', 'Left', 'Right']
    
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action_idx):
        """
        Take action, return (next_state, reward, done)
        """
        action = self.actions[action_idx]
        next_state = (self.state[0] + action[0], self.state[1] + action[1])
        
        # Check boundaries
        if (next_state[0] < 0 or next_state[0] >= self.size or
            next_state[1] < 0 or next_state[1] >= self.size):
            next_state = self.state  # Stay in place
            reward = -1
        # Check obstacles
        elif next_state in self.obstacles:
            next_state = self.state  # Stay in place
            reward = -1
        # Check goal
        elif next_state == self.goal:
            reward = 10
        else:
            reward = -0.1
        
        self.state = next_state
        done = (next_state == self.goal)
        
        return next_state, reward, done
    
    def visualize(self, values=None, policy=None):
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # Draw grid
        for i in range(self.size):
            for j in range(self.size):
                # Color based on value
                if values is not None:
                    value = values.get((i, j), 0)
                    color = plt.cm.RdYlGn((value + 1) / 11)  # Normalize
                else:
                    color = 'white'
                
                rect = Rectangle((j, self.size-1-i), 1, 1, 
                                facecolor=color, edgecolor='black', linewidth=2)
                ax.add_patch(rect)
                
                # Mark obstacles
                if (i, j) in self.obstacles:
                    ax.text(j+0.5, self.size-1-i+0.5, 'X', 
                           ha='center', va='center', fontsize=20, fontweight='bold')
                # Mark goal
                elif (i, j) == self.goal:
                    ax.text(j+0.5, self.size-1-i+0.5, 'G', 
                           ha='center', va='center', fontsize=20, 
                           fontweight='bold', color='green')
                # Show value
                elif values is not None:
                    ax.text(j+0.5, self.size-1-i+0.5, f'{value:.1f}', 
                           ha='center', va='center', fontsize=10)
                
                # Show policy arrow
                if policy is not None and (i, j) not in self.obstacles and (i, j) != self.goal:
                    action_idx = policy.get((i, j), 0)
                    arrows = ['↑', '↓', '←', '→']
                    ax.text(j+0.5, self.size-1-i+0.2, arrows[action_idx], 
                           ha='center', va='center', fontsize=16, color='blue')
        
        ax.set_xlim(0, self.size)
        ax.set_ylim(0, self.size)
        ax.set_aspect('equal')
        ax.axis('off')
        plt.tight_layout()
        return fig

# Create and visualize environment
env = GridWorld(size=5)
fig = env.visualize()
plt.title('GridWorld Environment', fontsize=14, fontweight='bold', pad=20)
plt.show()

print("Environment created!")
print(f"Grid size: {env.size}x{env.size}")
print(f"Start: (0, 0)")
print(f"Goal: {env.goal}")
print(f"Obstacles: {env.obstacles}")

## 14.2 Value Iteration

In [ ]:
def value_iteration(env, gamma=0.9, theta=0.001, max_iterations=1000):
    """
    Value Iteration algorithm
    """
    # Initialize value function
    V = defaultdict(float)
    
    for iteration in range(max_iterations):
        delta = 0
        
        # For each state
        for i in range(env.size):
            for j in range(env.size):
                state = (i, j)
                
                # Skip terminal and obstacle states
                if state == env.goal or state in env.obstacles:
                    continue
                
                v = V[state]
                
                # Compute value for each action
                action_values = []
                for action_idx in range(len(env.actions)):
                    env.state = state
                    next_state, reward, done = env.step(action_idx)
                    action_value = reward + gamma * V[next_state]
                    action_values.append(action_value)
                
                # Bellman update: take max over actions
                V[state] = max(action_values)
                delta = max(delta, abs(v - V[state]))
        
        if delta < theta:
            print(f"Value iteration converged in {iteration+1} iterations")
            break
    
    # Extract policy
    policy = {}
    for i in range(env.size):
        for j in range(env.size):
            state = (i, j)
            if state == env.goal or state in env.obstacles:
                continue
            
            action_values = []
            for action_idx in range(len(env.actions)):
                env.state = state
                next_state, reward, done = env.step(action_idx)
                action_value = reward + gamma * V[next_state]
                action_values.append(action_value)
            
            policy[state] = np.argmax(action_values)
    
    return V, policy

# Run value iteration
V_star, policy_star = value_iteration(env, gamma=0.9)

# Visualize results
fig = env.visualize(values=V_star, policy=policy_star)
plt.title('Value Iteration Results\n(Values and Optimal Policy)', 
         fontsize=14, fontweight='bold', pad=20)
plt.show()

print("\nOptimal Value Function:")
for i in range(env.size):
    for j in range(env.size):
        print(f"{V_star[(i,j)]:6.2f}", end=" ")
    print()

## 14.3 Q-Learning

In [ ]:
class QLearningAgent:
    def __init__(self, n_actions, learning_rate=0.1, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        self.lr = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_actions = n_actions
    
    def get_action(self, state, training=True):
        """Epsilon-greedy action selection"""
        if training and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.Q[state])
    
    def update(self, state, action, reward, next_state, done):
        """Q-learning update"""
        current_q = self.Q[state][action]
        
        if done:
            target_q = reward
        else:
            target_q = reward + self.gamma * np.max(self.Q[next_state])
        
        # Q-learning update
        self.Q[state][action] = current_q + self.lr * (target_q - current_q)

# Train Q-learning agent
agent = QLearningAgent(n_actions=4, learning_rate=0.1, gamma=0.9, epsilon=0.1)

n_episodes = 1000
episode_rewards = []
episode_lengths = []

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    steps = 0
    
    for step in range(100):  # Max 100 steps per episode
        action = agent.get_action(state, training=True)
        next_state, reward, done = env.step(action)
        
        agent.update(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
        steps += 1
        
        if done:
            break
    
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    
    if (episode + 1) % 200 == 0:
        avg_reward = np.mean(episode_rewards[-100:])
        avg_length = np.mean(episode_lengths[-100:])
        print(f"Episode {episode+1}: Avg Reward={avg_reward:.2f}, Avg Steps={avg_length:.1f}")

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Smooth rewards
window = 50
smoothed_rewards = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
smoothed_lengths = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')

axes[0].plot(smoothed_rewards, linewidth=2)
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Total Reward', fontsize=12)
axes[0].set_title('Q-Learning: Reward per Episode', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(smoothed_lengths, linewidth=2, color='orange')
axes[1].set_xlabel('Episode', fontsize=12)
axes[1].set_ylabel('Steps to Goal', fontsize=12)
axes[1].set_title('Q-Learning: Episode Length', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14.4 Learned Q-Values and Policy

In [ ]:
# Extract learned values and policy
learned_values = {}
learned_policy = {}

for i in range(env.size):
    for j in range(env.size):
        state = (i, j)
        if state not in env.obstacles and state != env.goal:
            learned_values[state] = np.max(agent.Q[state])
            learned_policy[state] = np.argmax(agent.Q[state])

# Visualize
fig = env.visualize(values=learned_values, policy=learned_policy)
plt.title('Q-Learning Results\n(Learned Values and Policy)', 
         fontsize=14, fontweight='bold', pad=20)
plt.show()

# Test learned policy
print("\nTesting Learned Policy:")
state = env.reset()
path = [state]

for step in range(20):
    action = agent.get_action(state, training=False)
    next_state, reward, done = env.step(action)
    path.append(next_state)
    
    print(f"Step {step+1}: {state} → {env.action_names[action]} → {next_state} (reward={reward:.1f})")
    
    state = next_state
    if done:
        print(f"\nReached goal in {step+1} steps!")
        break

## Key Takeaways

### 1. **Markov Decision Process (MDP)**
- Framework for sequential decision making
- Components: States, Actions, Transitions, Rewards
- Markov property: Future depends only on current state
- Goal: Find policy π that maximizes cumulative reward

### 2. **Value Functions**
- **State value**: V(s) = Expected cumulative reward from state s
- **Action value**: Q(s,a) = Expected reward from state s, action a
- Bellman equation: Recursive relationship
- Optimal policy: Choose action with max Q-value

### 3. **Value Iteration**
- Model-based: Requires knowledge of transitions P(s'|s,a)
- Iteratively update V(s) using Bellman equation
- Converges to optimal value function V*
- Extract policy: π(s) = argmax_a Q(s,a)

### 4. **Q-Learning**
- Model-free: Learns from experience (no transition model)
- Off-policy: Learns optimal Q* while following ε-greedy
- Update rule: Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') - Q(s,a)]
- Exploration vs exploitation: ε-greedy balances both

### 5. **Key Differences**
| Aspect | Value Iteration | Q-Learning |
|--------|----------------|------------|
| Model | Required | Not required |
| Updates | All states | Visited states |
| Convergence | Guaranteed | Guaranteed (with conditions) |
| Speed | Fast with small state space | Slow, needs many episodes |

### 6. **Exploration Strategies**
- **ε-greedy**: Random action with probability ε
- **Softmax**: Sample from Boltzmann distribution
- **Upper Confidence Bound**: Bonus for rarely visited states
- Exploration essential for finding optimal policy

### 7. **Extensions**
- **SARSA**: On-policy TD learning
- **Double Q-Learning**: Reduces overestimation bias
- **DQN**: Deep neural network for Q-function
- **Policy Gradient**: Directly optimize policy
- **Actor-Critic**: Combine value and policy methods
- **A3C, PPO, SAC**: Modern RL algorithms

### 8. **Practical Tips**
- Start with simple environments (GridWorld)
- Tune learning rate α and discount γ
- Balance exploration (ε) carefully
- Use experience replay for stability
- Normalize rewards if possible

## References

1. "Reinforcement Learning: An Introduction" - Sutton & Barto (2nd Edition)
2. "Playing Atari with Deep Reinforcement Learning" - Mnih et al. (2013)
3. CS229 Lecture Notes on Reinforcement Learning
4. David Silver's RL Course (UCL)

---

**Congratulations! You've completed all 14 CS229 lectures!** 🎉